In [15]:
cleaned_customer_transaction_data = pd.read_csv(r"C:\Users\user\Downloads\Fraudulent_Transaction_Detection_for_Finlora_Company\Finlora_Dataset\artifacts\Cleaned_Data.csv")
from xgboost import XGBClassifier

!pip install xgboost
!pip install xgboost

from xgboost import XGBClassifier


NameError: name 'pd' is not defined

In [ ]:
cleaned_customer_transaction_data['timestamp'] = pd.to_datetime(
    cleaned_customer_transaction_data['timestamp'],
    errors='coerce'
)

cleaned_customer_transaction_data['hour'] = cleaned_customer_transaction_data['timestamp'].dt.hour
cleaned_customer_transaction_data['day_of_week'] = cleaned_customer_transaction_data['timestamp'].dt.day_name()
cleaned_customer_transaction_data['is_weekend'] = cleaned_customer_transaction_data['day_of_week'].isin(['Saturday','Sunday']).astype(int)
cleaned_customer_transaction_data['month'] = cleaned_customer_transaction_data['timestamp'].dt.month_name()


In [ ]:
cleaned_customer_transaction_data.columns


Index(['Unnamed: 0', 'transaction_id', 'customer_id', 'timestamp',
       'home_country', 'source_currency', 'dest_currency', 'channel',
       'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest',
       'device_id', 'new_device', 'ip_address', 'ip_country',
       'location_mismatch', 'ip_risk_score', 'kyc_tier', 'account_age_days',
       'device_trust_score', 'chargeback_history_count', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'home_country_norm', 'ip_country_norm', 'kyc_tier_norm',
       'country_mismatch', 'hour', 'day_of_week', 'is_weekend', 'month'],
      dtype='object')

In [ ]:
cleaned_customer_transaction_data.columns


Index(['Unnamed: 0', 'transaction_id', 'customer_id', 'timestamp',
       'home_country', 'source_currency', 'dest_currency', 'channel',
       'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest',
       'device_id', 'new_device', 'ip_address', 'ip_country',
       'location_mismatch', 'ip_risk_score', 'kyc_tier', 'account_age_days',
       'device_trust_score', 'chargeback_history_count', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'home_country_norm', 'ip_country_norm', 'kyc_tier_norm',
       'country_mismatch', 'hour', 'day_of_week', 'is_weekend', 'month'],
      dtype='object')

In [ ]:
cleaned_customer_transaction_data.columns


Index(['Unnamed: 0', 'transaction_id', 'customer_id', 'timestamp',
       'home_country', 'source_currency', 'dest_currency', 'channel',
       'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest',
       'device_id', 'new_device', 'ip_address', 'ip_country',
       'location_mismatch', 'ip_risk_score', 'kyc_tier', 'account_age_days',
       'device_trust_score', 'chargeback_history_count', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'home_country_norm', 'ip_country_norm', 'kyc_tier_norm',
       'country_mismatch', 'hour', 'day_of_week', 'is_weekend', 'month'],
      dtype='object')

In [ ]:
# Creating threshold‑based features from key risk signals
cleaned_customer_transaction_data['timestamp'] = pd.to_datetime(
    cleaned_customer_transaction_data['timestamp']
)

cleaned_customer_transaction_data['late_night_hours'] = (
    (cleaned_customer_transaction_data['hour'] > 3) &
    (cleaned_customer_transaction_data['hour'] < 8)
).astype(int)

cleaned_customer_transaction_data['amount_high'] = (
    cleaned_customer_transaction_data['amount_usd'] > 1000
).astype(int)

cleaned_customer_transaction_data['high_ip_risk'] = (
    cleaned_customer_transaction_data['ip_risk_score'] > 0.8
).astype(int)

cleaned_customer_transaction_data['low_device_trust'] = (
    cleaned_customer_transaction_data['device_trust_score'] < 0.5
).astype(int)

cleaned_customer_transaction_data['new_account'] = (
    cleaned_customer_transaction_data['account_age_days'] < 30
).astype(int)

# Correct velocity feature
cleaned_customer_transaction_data['velocity_spike'] = (
    cleaned_customer_transaction_data['txn_velocity_1h'] > 3
).astype(int)

high_risk_signal_features = cleaned_customer_transaction_data[
    [
        'late_night_hours',
        'amount_high',
        'high_ip_risk',
        'low_device_trust',
        'new_account',
        'velocity_spike'
    ]
]

high_risk_signal_features.head()


,late_night_hours,amount_high,high_ip_risk,low_device_trust,new_account,velocity_spike
0,0,0,0,0,0,0
1,0,0,0,1,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0


In [ ]:
# Checking the features in our dataset
list(cleaned_customer_transaction_data.columns)


['Unnamed: 0',
 'transaction_id',
 'customer_id',
 'timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'exchange_rate_src_to_dest',
 'device_id',
 'new_device',
 'ip_address',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'chargeback_history_count',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud',
 'home_country_norm',
 'ip_country_norm',
 'kyc_tier_norm',
 'country_mismatch',
 'hour',
 'day_of_week',
 'is_weekend',
 'month',
 'late_night_hours',
 'amount_high',
 'high_ip_risk',
 'low_device_trust',
 'new_account',
 'velocity_spike']

In [ ]:
# Feature selection pipeline

# Dropping identifier columns (IDs)
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(
    ['transaction_id', 'customer_id', 'device_id', 'ip_address'],
    axis=1
)

# Dropping irrelevant variables
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(
    ['chargeback_history_count', 'month', 'exchange_rate_src_to_dest', 'Unnamed: 0'],
    axis=1
)


Categorical features

Machine‑learning models cannot understand text categories.
You must identify them so you can encode them properly.

In [ ]:
categorical_features = cleaned_customer_transaction_data.select_dtypes(
    include=['object','bool']
).columns

categorical_features


Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'new_device', 'ip_country', 'location_mismatch', 'kyc_tier',
       'home_country_norm', 'ip_country_norm', 'kyc_tier_norm',
       'country_mismatch', 'day_of_week'],
      dtype='object')

In [ ]:
# Defining numerical features
numerical_features = cleaned_customer_transaction_data.select_dtypes(
    include=['int', 'float']
).columns.drop('is_fraud')

numerical_features


Index(['amount_src', 'amount_usd', 'fee', 'ip_risk_score', 'account_age_days',
       'device_trust_score', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'hour', 'is_weekend',
       'late_night_hours', 'amount_high', 'high_ip_risk', 'low_device_trust',
       'new_account', 'velocity_spike'],
      dtype='object')

In [ ]:

print(f"Categorical: {len(categorical_features)}")
print(f"Numerical: {len(numerical_features)}")
print(f"Dataset: {cleaned_customer_transaction_data.shape}")


Categorical: 13
Numerical: 18
Dataset: (10840, 33)


In [ ]:
cleaned_customer_transaction_data.columns



Index(['timestamp', 'home_country', 'source_currency', 'dest_currency',
       'channel', 'amount_src', 'amount_usd', 'fee', 'new_device',
       'ip_country', 'location_mismatch', 'ip_risk_score', 'kyc_tier',
       'account_age_days', 'device_trust_score', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'home_country_norm', 'ip_country_norm', 'kyc_tier_norm',
       'country_mismatch', 'hour', 'day_of_week', 'is_weekend',
       'late_night_hours', 'amount_high', 'high_ip_risk', 'low_device_trust',
       'new_account', 'velocity_spike'],
      dtype='object')

In [ ]:
cleaned_customer_transaction_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10840 entries, 0 to 10839
Data columns (total 33 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   timestamp            10840 non-null  datetime64[ns, UTC]
 1   home_country         10840 non-null  object             
 2   source_currency      10840 non-null  object             
 3   dest_currency        10840 non-null  object             
 4   channel              10804 non-null  object             
 5   amount_src           10840 non-null  float64            
 6   amount_usd           10840 non-null  float64            
 7   fee                  10840 non-null  float64            
 8   new_device           10840 non-null  bool               
 9   ip_country           10840 non-null  object             
 10  location_mismatch    10840 non-null  bool               
 11  ip_risk_score        10840 non-null  float64            
 12  kyc_tier          

In [18]:
import pandas as pd

cleaned_customer_transaction_data = pd.read_csv("../Finlora_Dataset/artifacts/Cleaned_Data.csv")


In [19]:
cleaned_customer_transaction_data.to_csv(
    "../Finlora_Dataset/artifacts/Engineered_Data.csv",
    index=False
)


In [38]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Select X and y
target_col = 'is_fraud'
drop_cols = ['Unnamed: 0', 'transaction_id', 'customer_id', 'timestamp', 'device_id', 'ip_address', target_col]

X = df.drop(columns=[col for col in drop_cols if col in df.columns])
y = df[target_col]

# 2. Define Preprocessor
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Transform Features
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 5. Calculate Scale Pos Weight & Train
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    random_state=42, 
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.1, 
    scale_pos_weight=scale_pos_weight, 
    eval_metric='logloss'
)
xgb_model.fit(X_train_processed, y_train)

# 6. Predict & Evaluate
y_pred_xgb = xgb_model.predict(X_test_processed)
y_proba_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

print("XGBoost Results:")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Genuine', 'Fraud']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))
print(f"\nROC_AUC Score: {roc_auc_score(y_test, y_proba_xgb):.4f}")

XGBoost Results:

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.98      0.99      0.99      1971
       Fraud       0.93      0.82      0.87       197

    accuracy                           0.98      2168
   macro avg       0.95      0.91      0.93      2168
weighted avg       0.98      0.98      0.98      2168


Confusion Matrix:
[[1958   13]
 [  35  162]]

ROC_AUC Score: 0.9536
